<a href="https://colab.research.google.com/github/Melonangie/Galaxy-Vision/blob/main/EDA_Starlight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install specutils astropy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 12.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.6/984.6 kB 37.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [specutils]


In [2]:
import glob
from astropy.io import fits
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from astropy.table import Table
from astropy.stats import sigma_clipped_stats
from scipy.integrate import trapezoid
from scipy.stats import kruskal, spearmanr

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/MyDrive/Proyecto_morfologia/data/morfologia.fits" /content/
!cp "/content/drive/MyDrive/Proyecto_morfologia/data/MPL-11.zip" /content/
!cp "/content/drive/MyDrive/Proyecto_morfologia/data/FoV-OUT.zip" /content/

!unzip -o -q /content/MPL-11.zip -d /content/fits
!unzip -o -q /content/FoV-OUT.zip -d /content/out

!df -h /content

!ls /content
!ls /content/fits/MPL-11 | wc -l
!ls /content/out/FoV-OUT | wc -l

%cd /content/fits/MPL-11
!rename 's/-1D.fits.gz//' *.fits.gz
!ls -U | head -4

%cd /content/out/FoV-OUT
!rename 's/-1D.out//' *.out
!ls -U | head -4

Mounted at /content/drive
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   27G   82G  25% /
drive  fits  FoV-OUT.zip  morfologia.fits  MPL-11.zip  out  sample_data
10243
10243
/content/fits/MPL-11
manga-11007-1902
manga-12489-6102
manga-8315-12705
manga-8945-12705
/content/out/FoV-OUT
manga-11948-12703-1D.out
manga-8547-12705-1D.out
manga-8604-12701-1D.out
manga-8329-12701-1D.out


In [ ]:
FITS_DIR   = "/content/fits/MPL-11"  # one 1D .fits spectrum per galaxy
OUT_DIR    = "/content/out/FoV-OUT"   # STARLIGHT outputs
MORPH_FILE = "/content/morfologia.fits"

fits_files = sorted(glob.glob(f"{FITS_DIR}/*.fits.gz"))
# STARLIGHT outputs may be called *.out or *-out.txt -> accept both
out_files  = sorted(glob.glob(f"{OUT_DIR}/*.out"))

print(len(fits_files), "FITS spectra, ", len(out_files), "STARLIGHT outputs")

10243 FITS spectra,  10243 STARLIGHT outputs


In [ ]:
data=fit[1].data
data

FITS_rec([( 3622. ,  3.25357866, 22.82295609, 0.1129891 , 46.52426147),
          ( 3623.5,  3.44680023, 21.68067741, 0.10931516, 46.53480911),
          ( 3625. ,  3.3696146 , 25.60529709, 0.10779376, 46.51317215),
          ...,
          (10348. , 11.00659084, 34.49480057, 0.14804165, 47.11366272),
          (10349.5, 10.69810963, 26.02185631, 0.16736087, 47.11603928),
          (10351. ,  9.88376045, 23.47935295, 0.16497421, 47.11573029)],
         dtype=(numpy.record, [('WAVE', '>f8'), ('FLUX_1RE', '>f8'), ('FLUX_FoV', '>f8'), ('ERROR_1RE', '>f8'), ('ERROR_FoV', '>f8')]))

In [ ]:
fit[1].header

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                   40 / length of dimension 1                          
NAXIS2  =                 4487 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                    5 / number of table fields                         
TTYPE1  = 'WAVE    '                                                            
TFORM1  = 'D       '                                                            
TTYPE2  = 'FLUX_1RE'                                                            
TFORM2  = 'D       '                                                            
TTYPE3  = 'FLUX_FoV'        

In [ ]:
flux=data['FLUX_1RE']
wave=['wave']
spec=Spectrum1D(spectral_axis=wave,flux=flux)
spec.spectral_axis, spec.flux

/tmp/ipykernel_1344/2428141908.py:3: AstropyDeprecationWarning: The Spectrum1D class is deprecated and may be removed in a future version.
        Use Spectrum instead.
  spec=Spectrum1D(spectral_axis=wave,flux=flux)


ValueError: Flux must be a `Quantity` object.